## LRModelQ2

In [1]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import joblib


PART_B_DIR = Path.cwd().parents[0]
DATA_FILE = PART_B_DIR / "data" / "yelp_clean.csv"
MODEL_DIR = PART_B_DIR / "models"

### Data Preparation

In [2]:
# --- Load Clean Data (Yelp 3-class sentiment) ---
df = pd.read_csv(DATA_FILE)

In [3]:
# --- Split into train/test ---
# stratify keeps the class ratio in both halves; random_state=42 is reused in Q3 and Q4.
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['sentiment'],
    test_size=0.2, random_state=42, stratify=df['sentiment']
)

print("Training reviews:", len(X_train))
print("Test reviews    :", len(X_test))

Training reviews: 23998
Test reviews    : 6000


### Baseline Model Training

In [4]:
# --- Build Logistic Regression pipeline ---
# One Pipeline so the vectoriser only ever sees training data, no leakage.
# Shared TF-IDF config across all 4 Q2 models. class_weight balances the smaller neutral class.
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1, 2),
                              min_df=2, sublinear_tf=True, strip_accents="unicode")),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced',
                               random_state=42))
])

In [5]:
# --- Train, save for future use, predict based on test data ---
pipeline.fit(X_train, y_train)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(pipeline, MODEL_DIR / 'lr_pipeline.joblib')
preds = pipeline.predict(X_test)

### Model Evaluation

In [6]:
# --- Report base performance measures ---
# Macro average matters here because neutral is half the size of the other two classes.
print("=== Q2: Logistic Regression Classification Report (Yelp 3-class sentiment) ===")
print(classification_report(y_test, preds, digits=4))

=== Q2: Logistic Regression Classification Report (Yelp 3-class sentiment) ===
              precision    recall  f1-score   support

    negative     0.8495    0.7975    0.8227      2400
     neutral     0.4747    0.5950    0.5281      1200
    positive     0.8547    0.7987    0.8258      2400

    accuracy                         0.7575      6000
   macro avg     0.7263    0.7304    0.7255      6000
weighted avg     0.7766    0.7575    0.7650      6000

